# Lab 57 (solution): Real durable backends, integration-tested

Reference implementation. [Lab 54](../../54-production-durable-backends/) used hand-written fakes to show the lease/ack/reclaim contract. This lab uses the actual `redis-py` Streams and `boto3` SQS APIs, exercised by `fakeredis` and `moto` (which run the real commands), with a `test_integration.py` that runs the same backends against a live Redis or LocalStack.

## Step 0: Setup

In [ ]:
import boto3
import fakeredis
from moto import mock_aws
from backends import RedisStreamsBackend, SQSBackend, run_contract, _make_send
# Lab 54 used hand-written fakes to show the contract. Here the lab code issues REAL redis-py and
# boto3 calls; fakeredis and moto run those commands in-process. The same code runs against a live
# server in test_integration.py.
print("backends issue real XADD/XREADGROUP/XACK/XAUTOCLAIM and SendMessage/ReceiveMessage/DeleteMessage")

## Step 1: Real Redis Streams (via fakeredis)

In [ ]:
# Real redis-py Streams consumer group, exercised by fakeredis (which implements the actual
# Stream commands). XGROUP_CREATE on construction; XAUTOCLAIM reclaims unacked entries; the PEL
# delivery count (XPENDING) drives give-up to a dead stream.
r = fakeredis.FakeStrictRedis(decode_responses=True)
redis_state = run_contract(RedisStreamsBackend(r), _make_send())
print("redis-streams:", redis_state)

## Step 2: Real SQS with a server-side redrive policy (via moto)

In [ ]:
# Real boto3 SQS, exercised by moto. The give-up path is the queue's OWN redrive policy
# (maxReceiveCount -> dead-letter queue) - SQS enforces it server-side, not the lab code.
with mock_aws():
    sqs = boto3.client("sqs", region_name="us-east-1")
    sqs_state = run_contract(SQSBackend.create(sqs, max_receives=3), _make_send())
print("sqs:          ", sqs_state)
print("\nidentical end state from the real client libraries - and from two give-up mechanisms")
print("(PEL delivery count vs SQS redrive policy).")

## Step 3: The same code against live infrastructure

In [ ]:
# The identical backends run against live infrastructure - only the injected client changes:
print("Redis:      RedisStreamsBackend(redis.Redis.from_url(os.environ['REDIS_URL']))")
print("LocalStack: SQSBackend.create(boto3.client('sqs', endpoint_url=os.environ['AWS_ENDPOINT_URL']))")
print()
print("$ REDIS_URL=redis://localhost:6379 pytest test_integration.py")
print("$ AWS_ENDPOINT_URL=http://localhost:4566 pytest test_integration.py   # LocalStack")
print("Each test skips when its service is not configured, so the suite is safe to run anywhere.")

## What you built

The Lab 54 contract on the real client libraries, integration-tested. `backends.py` is now shippable code: `RedisStreamsBackend` issues real `redis-py` `XADD` / `XREADGROUP` / `XACK` / `XAUTOCLAIM` against a consumer group, and `SQSBackend` issues real `boto3` `SendMessage` / `ReceiveMessage` / `DeleteMessage` against a queue whose redrive policy enforces give-up server-side. The self-test exercises that exact code through `fakeredis` and `moto`, which run the real commands in-process; `test_integration.py` runs it against a live Redis or LocalStack and skips when neither is set. Both backends reach the identical end state - m0 acked, m1 recovered on redelivery, m2 in the dead set - from two unrelated give-up mechanisms (the consumer group's Pending Entries List vs SQS's `maxReceiveCount` redrive).

**Where this simplifies:** the self-test uses `min_idle_time=0` for `XAUTOCLAIM` and `VisibilityTimeout=0` for SQS so reclaim is immediate and deterministic offline; in production (and in the live integration test) you set the idle time / visibility timeout to the real lease duration and let it lapse. `fakeredis` and `moto` mirror the documented semantics but not every server edge case - the integration test against real infrastructure is what closes that gap, which is exactly why it ships with the lab. At-least-once delivery still means the consumer must be idempotent, and a real deployment configures stream `MAXLEN`, consumer-group lag monitoring, and the SQS DLQ alarm.